In [0]:
# The missing piece: Importing the Spark SQL functions
from pyspark.sql.functions import col, sum as spark_sum, count, round as spark_round, when

print("1. Loading Cleaned Data from Silver Schema...")

# Dimensions
silver_customer = spark.table("medallion.silver.customer_data")
silver_product = spark.table("medallion.silver.product_master")
silver_store = spark.table("medallion.silver.store_master")

# Facts
silver_sales = spark.table("medallion.silver.sales_transactions")
silver_inventory = spark.table("medallion.silver.inventory_data")
silver_clickstream = spark.table("medallion.silver.clickstream_events")

1. Loading Cleaned Data from Silver Schema...


In [0]:
print("2. Writing Dimension Tables (Lookup Tables)...")

silver_customer.write.format("delta").mode("overwrite").saveAsTable("medallion.gold.dim_customer")
silver_product.write.format("delta").mode("overwrite").saveAsTable("medallion.gold.dim_product")
silver_store.write.format("delta").mode("overwrite").saveAsTable("medallion.gold.dim_store")

print("-> dim_customer, dim_product, and dim_store are created.")

2. Writing Dimension Tables (Lookup Tables)...
-> dim_customer, dim_product, and dim_store are created.


In [0]:
print("3. Building Enriched fact_sales Table...")

fact_sales_df = silver_sales.alias("s") \
    .join(spark.table("medallion.gold.dim_product").alias("p"), col("s.product_id") == col("p.product_id"), "left") \
    .join(spark.table("medallion.gold.dim_store").alias("st"), col("s.store_id") == col("st.store_id"), "left") \
    .join(spark.table("medallion.gold.dim_customer").alias("c"), col("s.customer_id") == col("c.customer_id"), "left") \
    .select("s.*", "p.brand", "p.category", "st.city", "st.state", "c.loyalty_status")

fact_sales_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("medallion.gold.fact_sales")

print("-> fact_sales created (left joins — no sales records dropped).")

3. Building Enriched fact_sales Table...
-> fact_sales created (left joins — no sales records dropped).


In [0]:
print("4. Processing Secondary Facts...")

# Inventory Fact (left joins to keep all inventory records)
fact_inventory_df = silver_inventory.alias("i") \
    .join(spark.table("medallion.gold.dim_product").alias("p"), "product_id", "left") \
    .join(spark.table("medallion.gold.dim_store").alias("st"), "store_id", "left") \
    .select("i.*", "p.brand", "st.city", "st.state")
fact_inventory_df.write.format("delta").mode("overwrite").saveAsTable("medallion.gold.fact_inventory")

# Clickstream Fact (overwriteSchema handles the event_timestamp date→timestamp change)
fact_clickstream_df = silver_clickstream.alias("clk") \
    .join(spark.table("medallion.gold.dim_product").alias("p"), "product_id", "left") \
    .select("clk.*", "p.brand", "p.category")
fact_clickstream_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medallion.gold.fact_clickstream")

print("-> fact_inventory and fact_clickstream created.")

4. Processing Secondary Facts...
-> fact_inventory and fact_clickstream created.


In [0]:
print("5. Creating Aggregated KPI Tables...")

# Channel-wise Performance
channel_agg = spark.table("medallion.gold.fact_sales").groupBy("channel").agg(
    count("transaction_id").alias("total_orders"),
    spark_round(spark_sum("net_sales"), 2).alias("total_revenue"),
    spark_round(spark_sum("gross_margin"), 2).alias("total_profit")
)
channel_agg.write.format("delta").mode("overwrite").saveAsTable("medallion.gold.agg_channel_performance")

# Store-wise Performance (using col for the order by)
store_agg = spark.table("medallion.gold.fact_sales").groupBy("city", "state").agg(
    spark_round(spark_sum("net_sales"), 2).alias("total_revenue")
).orderBy(col("total_revenue").desc())
store_agg.write.format("delta").mode("overwrite").saveAsTable("medallion.gold.agg_store_performance")

print("-> KPI tables ready.")

5. Creating Aggregated KPI Tables...
-> KPI tables ready.


In [0]:
%sql
SELECT * FROM medallion.gold.agg_channel_performance;

channel,total_orders,total_revenue,total_profit
Store,332602,6.7116574E8,1.7496010941E8
Online,332998,6.7032288385E8,1.7475250663E8
Mobile,334300,6.7313659428E8,1.7558814322E8


In [0]:
from pyspark.sql.functions import avg as spark_avg

print("6. Advanced BI Aggregations...")

# ── 1. Customer Loyalty ROI ──────────────────────────────────────────────────
# Business Q: Are Gold members more profitable per transaction than Standard?
fact_sales = spark.table("medallion.gold.fact_sales")

agg_loyalty = fact_sales.groupBy("loyalty_status").agg(
    count("transaction_id").alias("total_orders"),
    spark_round(spark_sum("net_sales"), 2).alias("total_revenue"),
    spark_round(spark_avg("gross_margin"), 2).alias("average_margin_per_order")
).orderBy(col("total_revenue").desc())

agg_loyalty.write.format("delta").mode("overwrite").saveAsTable("medallion.gold.agg_loyalty_performance")
print("-> agg_loyalty_performance created.")

6. Advanced BI Aggregations...
-> agg_loyalty_performance created.


In [0]:
%sql
-- Customer Loyalty ROI: Gold vs Standard profitability per order
SELECT * FROM medallion.gold.agg_loyalty_performance
ORDER BY total_revenue DESC;

loyalty_status,total_orders,total_revenue,average_margin_per_order
Silver,252519,5.0935103336E8,525.66
Standard,250730,5.0452311385E8,524.81
Gold,248968,5.0211110771E8,526.28
Bronze,247683,4.9863996321E8,524.65


In [0]:
# ── 2. Regional Product Demand ───────────────────────────────────────────────
# Business Q: Which categories dominate in Maharashtra vs Karnataka?
agg_regional = fact_sales.groupBy("state", "category").agg(
    spark_round(spark_sum("net_sales"), 2).alias("total_net_sales")
).orderBy(col("total_net_sales").desc())

agg_regional.write.format("delta").mode("overwrite").saveAsTable("medallion.gold.agg_regional_category_sales")
print("-> agg_regional_category_sales created.")

-> agg_regional_category_sales created.


In [0]:
%sql
-- Regional Product Demand: Top state-category combinations by revenue
SELECT * FROM medallion.gold.agg_regional_category_sales
ORDER BY total_net_sales DESC
LIMIT 15;

state,category,total_net_sales
Maharashtra,Electronics,2.2962988244E8
Maharashtra,Sports,1.6412258403E8
Maharashtra,Home,1.4224516241E8
Maharashtra,Beauty,1.0728692308E8
Delhi,Electronics,1.0579396981E8
Maharashtra,Clothing,1.0348893085E8
Rajasthan,Electronics,9.298038751E7
Delhi,Sports,7.515585518E7
Karnataka,Electronics,6.793782594E7
Rajasthan,Sports,6.631518648E7


In [0]:
from pyspark.sql.functions import avg as spark_avg

# ── 3. Inventory Turnover & Risk ─────────────────────────────────────────────
# Business Q: Which cities are overstocked on brands that don't sell there?
fact_inv = spark.table("medallion.gold.fact_inventory")

agg_inv_health = fact_inv.groupBy("city", "brand").agg(
    spark_round(spark_sum("stock_on_hand"), 0).alias("total_stock_on_hand"),
    spark_round(spark_avg("reorder_level"), 0).alias("avg_reorder_level")
).withColumn(
    "overstock_flag",
    when(col("total_stock_on_hand") > col("avg_reorder_level") * 3, "OVERSTOCK")
    .when(col("total_stock_on_hand") < col("avg_reorder_level"), "LOW STOCK")
    .otherwise("HEALTHY")
).orderBy(col("total_stock_on_hand").desc())

agg_inv_health.write.format("delta").mode("overwrite").saveAsTable("medallion.gold.agg_inventory_health")
print("-> agg_inventory_health created.")

-> agg_inventory_health created.


In [0]:
%sql
-- Inventory Health: Overstock risk by city and brand
SELECT * FROM medallion.gold.agg_inventory_health
WHERE overstock_flag = 'OVERSTOCK'
ORDER BY total_stock_on_hand DESC
LIMIT 15;

city,brand,total_stock_on_hand,avg_reorder_level,overstock_flag
Mumbai,SAMSUNG,24441,56.0,OVERSTOCK
Mumbai,APPLE,23538,55.0,OVERSTOCK
Mumbai,PUMA,19941,46.0,OVERSTOCK
Mumbai,NIKE,19355,50.0,OVERSTOCK
Mumbai,SONY,17804,59.0,OVERSTOCK
Delhi,SONY,17449,53.0,OVERSTOCK
Mumbai,LG,15980,55.0,OVERSTOCK
Delhi,PUMA,15727,52.0,OVERSTOCK
Jaipur,LG,15696,57.0,OVERSTOCK
Delhi,APPLE,15523,58.0,OVERSTOCK


In [0]:
# ── 4. Digital Behavior Analysis ──────────────────────────────────────────────
# Business Q: Which categories get the most digital engagement by event type?
fact_click = spark.table("medallion.gold.fact_clickstream")

agg_click = fact_click.groupBy("category", "event_type").agg(
    count("event_id").alias("total_events")
).orderBy("category", col("total_events").desc())

agg_click.write.format("delta").mode("overwrite").saveAsTable("medallion.gold.agg_clickstream_conversion")
print("-> agg_clickstream_conversion created.")
print("\n✅ All 4 advanced BI aggregations saved to medallion.gold.")

-> agg_clickstream_conversion created.

✅ All 4 advanced BI aggregations saved to medallion.gold.


In [0]:
%sql
-- Digital Behavior: Event engagement by category
SELECT category, event_type, total_events,
       ROUND(total_events * 100.0 / SUM(total_events) OVER (PARTITION BY category), 1) AS pct_of_category
FROM medallion.gold.agg_clickstream_conversion
ORDER BY category, total_events DESC;

category,event_type,total_events,pct_of_category
Beauty,view,84047,33.4
Beauty,add_to_cart,83821,33.3
Beauty,purchase,83523,33.2
Clothing,add_to_cart,83292,33.4
Clothing,view,83148,33.3
Clothing,purchase,83041,33.3
Electronics,add_to_cart,160605,33.4
Electronics,purchase,160235,33.4
Electronics,view,159410,33.2
Home,view,81020,33.4


In [0]:
# ─── KPI: Product & Category Performance ─────────────────────────────────────
from pyspark.sql.functions import col, count, sum as spark_sum, avg, round as spark_round, desc, countDistinct

print("Building product performance KPI...")

fact_sales = spark.table("medallion.gold.fact_sales")

agg_product = fact_sales \
    .groupBy("product_id", "category", "brand") \
    .agg(
        spark_round(spark_sum("net_sales"),    2).alias("total_revenue"),
        spark_round(spark_sum("gross_margin"), 2).alias("total_margin"),
        spark_sum("quantity").alias("total_units_sold"),
        count("transaction_id").alias("total_orders")
    ) \
    .orderBy(desc("total_revenue"))

agg_product.write.format("delta").mode("overwrite") \
    .saveAsTable("medallion.gold.agg_product_performance")

print("✅ agg_product_performance created")
display(agg_product.limit(10))

Building product performance KPI...
✅ agg_product_performance created


product_id,category,brand,total_revenue,total_margin,total_units_sold,total_orders
1218,Electronics,ADIDAS,5.946042988E7,1.40094062E7,65584,18746
1205,Home,NIKE,5.738544756E7,1.175906664E7,66062,18934
1476,Sports,NIKE,5.414755828E7,1.866656452E7,66096,18875
1304,Home,NIKE,5.321991781E7,1.683743059E7,66681,18996
1084,Electronics,SONY,4.679941215E7,1624040.37,66594,18945
1445,Sports,ADIDAS,4.646519959E7,2449959.83,65386,18745
1440,Electronics,ADIDAS,4.405734639E7,8179431.91,65684,18854
1302,Electronics,NIKE,4.289697674E7,9008345.48,66554,18982
1183,Clothing,NIKE,4.072062299E7,-661545.81,65870,18959
1404,Sports,SONY,3.97417076E7,9935603.86,65518,18701


In [0]:
# ─── KPI: Store Performance ───────────────────────────────────────────────────
print("Building store performance KPI...")

agg_store = fact_sales \
    .groupBy("store_id", "city", "state") \
    .agg(
        spark_round(spark_sum("net_sales"),    2).alias("total_revenue"),
        spark_round(spark_sum("gross_margin"), 2).alias("total_margin"),
        count("transaction_id").alias("total_orders"),
        spark_round(avg("net_sales"),          2).alias("avg_order_value")
    ) \
    .orderBy(desc("total_revenue"))

agg_store.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medallion.gold.agg_store_performance")

print("✅ agg_store_performance created")
display(agg_store.limit(10))

Building store performance KPI...
✅ agg_store_performance created


store_id,city,state,total_revenue,total_margin,total_orders,avg_order_value
68,Hyderabad,Telangana,2.059810417E7,5383910.96,10176,2024.18
73,Pune,Maharashtra,2.058989742E7,5290539.48,10193,2020.0
49,Delhi,Delhi,2.05839804E7,5344757.79,10173,2023.39
4,Delhi,Delhi,2.055308373E7,5321385.08,10061,2042.85
92,Bangalore,Karnataka,2.054227794E7,5286953.63,10117,2030.47
64,Delhi,Delhi,2.052205096E7,5342339.7,10153,2021.28
26,Mumbai,Maharashtra,2.051220092E7,5309497.56,10068,2037.37
38,Pune,Maharashtra,2.051016047E7,5383671.68,10102,2030.31
16,Chandigarh,Chandigarh,2.050143568E7,5343910.05,10078,2034.28
65,Delhi,Delhi,2.045839961E7,5347280.26,10024,2040.94


In [0]:
# ─── KPI: Monthly Revenue Trend ───────────────────────────────────────────────
from pyspark.sql.functions import date_format

print("Building monthly trend KPI...")

agg_monthly = fact_sales \
    .withColumn("year_month", date_format(col("order_date"), "yyyy-MM")) \
    .groupBy("year_month", "channel") \
    .agg(
        spark_round(spark_sum("net_sales"),    2).alias("monthly_revenue"),
        spark_round(spark_sum("gross_margin"), 2).alias("monthly_margin"),
        count("transaction_id").alias("monthly_orders")
    ) \
    .orderBy("year_month", "channel")

agg_monthly.write.format("delta").mode("overwrite") \
    .saveAsTable("medallion.gold.agg_monthly_trend")

print("✅ agg_monthly_trend created")
display(agg_monthly.limit(12))

Building monthly trend KPI...
✅ agg_monthly_trend created


year_month,channel,monthly_revenue,monthly_margin,monthly_orders
2024-01,Mobile,4.891304968E7,1.27463665E7,28258
2024-01,Online,4.866352536E7,1.263191814E7,28083
2024-01,Store,4.862401164E7,1.282192663E7,28076
2024-02,Mobile,4.548879686E7,1.186017947E7,26299
2024-02,Online,4.557063197E7,1.188893736E7,26506
2024-02,Store,4.547897767E7,1.178123143E7,26323
2024-03,Mobile,4.851699504E7,1.272262677E7,28199
2024-03,Online,4.86267797E7,1.267973112E7,28307
2024-03,Store,4.839815235E7,1.261858116E7,28018
2024-04,Mobile,4.694875912E7,1.22031716E7,27262


In [0]:
# ─── KPI: Inventory Stock-out Risk ────────────────────────────────────────────
print("Building inventory risk KPI...")

fact_inventory = spark.table("medallion.gold.fact_inventory")

agg_inventory = fact_inventory \
    .withColumn(
        "stock_status",
        col("stock_on_hand").cast("int")
    ) \
    .selectExpr(
        "product_id",
        "store_id",
        "brand",
        "city",
        "stock_on_hand",
        "reorder_level",
        """CASE 
            WHEN stock_on_hand = 0              THEN 'Out of Stock'
            WHEN stock_on_hand < reorder_level  THEN 'Below Reorder'
            ELSE                                     'Healthy'
           END AS stock_status"""
    )

agg_inventory.write.format("delta").mode("overwrite") \
    .saveAsTable("medallion.gold.agg_inventory_risk")

print("✅ agg_inventory_risk created")
display(
    agg_inventory.groupBy("stock_status")
    .count()
    .orderBy("stock_status")
)

Building inventory risk KPI...
✅ agg_inventory_risk created


stock_status,count
Below Reorder,276
Healthy,2200
Out of Stock,24


In [0]:
%sql
-- ─── GOVERNED SQL VIEWS ──────────────────────────────────────────────────────
-- These views are the ONLY layer that should be queried for reporting
-- Never query raw Silver or Gold tables directly from BI tools

CREATE OR REPLACE VIEW medallion.gold.v_channel_performance AS
  SELECT * FROM medallion.gold.agg_channel_performance;

CREATE OR REPLACE VIEW medallion.gold.v_product_performance AS
  SELECT * FROM medallion.gold.agg_product_performance;

CREATE OR REPLACE VIEW medallion.gold.v_store_performance AS
  SELECT * FROM medallion.gold.agg_store_performance;

CREATE OR REPLACE VIEW medallion.gold.v_monthly_trend AS
  SELECT * FROM medallion.gold.agg_monthly_trend;

CREATE OR REPLACE VIEW medallion.gold.v_loyalty_analysis AS
  SELECT * FROM medallion.gold.agg_loyalty_performance;

CREATE OR REPLACE VIEW medallion.gold.v_inventory_risk AS
  SELECT * FROM medallion.gold.agg_inventory_risk;

SELECT 'All Gold views created successfully' AS status;

status
All Gold views created successfully


In [0]:
%sql
-- ─── OPTIMIZE GOLD ───────────────────────────────────────────────────────────
OPTIMIZE medallion.gold.fact_sales
  ZORDER BY (order_date, channel, product_id);

OPTIMIZE medallion.gold.agg_product_performance
  ZORDER BY (category, brand);

OPTIMIZE medallion.gold.agg_store_performance
  ZORDER BY (city, state);

OPTIMIZE medallion.gold.agg_monthly_trend
  ZORDER BY (year_month, channel);

SELECT 'Gold OPTIMIZE complete' AS status;

status
Gold OPTIMIZE complete


In [0]:
%sql
-- ─── VERIFY ALL GOLD VIEWS ───────────────────────────────────────────────────
SHOW TABLES IN medallion.gold;

database,tableName,isTemporary
gold,agg_channel_performance,false
gold,agg_clickstream_conversion,false
gold,agg_inventory_health,false
gold,agg_inventory_risk,false
gold,agg_loyalty_performance,false
gold,agg_monthly_trend,false
gold,agg_product_performance,false
gold,agg_regional_category_sales,false
gold,agg_store_performance,false
gold,dim_customer,false


In [0]:
%sql
-- Pipeline monitoring dashboard view
CREATE OR REPLACE VIEW medallion.gold.v_pipeline_monitor AS
SELECT
    i.run_id,
    i.table_name,
    i.rows_loaded,
    i.loaded_at,
    i.status                AS ingestion_status
FROM medallion.bronze.ingestion_log i
ORDER BY i.loaded_at DESC;

SELECT 'Pipeline monitor view created' AS status;

status
Pipeline monitor view created
